# 2608.27372 — Ellipsoid-Fitting Phase Boundary

**Engineering Statements companion notebook — v3**

This version preserves three distinct computational readings:

\[
\boxed{\mathrm{SAT}\neq\mathrm{UNSAT}\neq\mathrm{UNRESOLVED}}
\]

A solver failure or ambiguous solver status is recorded as **UNRESOLVED**, rather than being counted as UNSAT.

The workflow is

\[
\text{asymptotic specification}
\rightarrow
\text{computation}
\rightarrow
\text{resolved or unresolved reading}.
\]

Source: arXiv:2608.27372  
Engineering Statement: `statements/2608-27372.yaml`

## 1. Runtime mode

The default is the report-scale Gaussian experiment:

\[
d=40,\qquad 8\text{ trials per density}.
\]

Use `"fast"` for a short validation run.

In [ ]:
MODE = "paper"   # "fast" or "paper"

if MODE == "fast":
    D = 12
    ALPHAS = [0.10, 0.15, 0.20, 0.23, 0.24, 0.25, 0.26, 0.27, 0.30, 0.35]
    TRIALS = 3
else:
    D = 40
    ALPHAS = [0.10, 0.15, 0.20, 0.225, 0.24, 0.25, 0.26, 0.275, 0.30, 0.35, 0.40]
    TRIALS = 8

SEED = 260827372

print({
    "mode": MODE,
    "d": D,
    "trials_per_density": TRIALS,
    "alpha_values": ALPHAS,
})

## 2. Install and import dependencies

In [ ]:
!pip -q install pyyaml cvxpy

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import yaml

print("cvxpy:", cp.__version__)
print("installed solvers:", cp.installed_solvers())

## 3. Load the Engineering Statement

The repository YAML is primary. The embedded fallback keeps the notebook portable.

In [ ]:
STATEMENT_PATH = Path("../statements/2608-27372.yaml")

fallback_yaml = """
id: 2608-27372
title: Universality and Sharp Thresholds for Ellipsoid Fitting
source:
  paper: https://arxiv.org/abs/2608.27372
objective: >
  Specify the ellipsoid-fitting problem through its constraint-density scaling,
  sharp SAT-UNSAT phase boundary, distribution-dependent threshold,
  and computational readings.
constraints:
  - Random vectors x_1,...,x_n lie in R^d.
  - Seek a positive-semidefinite matrix R such that x_i^T R x_i = 1 for every i.
  - Constraint density is alpha = n / d^2.
  - A sharp SAT-UNSAT phase boundary occurs at alpha_star(kappa).
  - The coordinate distribution enters the phase boundary through kappa = E[x_ij^4].
  - For Gaussian coordinates, kappa = 3 and alpha_star(3) = 1/4.
"""

if STATEMENT_PATH.exists():
    statement = yaml.safe_load(STATEMENT_PATH.read_text())
    print("Loaded:", STATEMENT_PATH)
else:
    statement = yaml.safe_load(fallback_yaml)
    print("Using embedded fallback statement.")

print(statement["title"])
print(statement["objective"].strip())

## 4. Gaussian specification

For Gaussian coordinates,

\[
\kappa=3,
\qquad
\alpha_\star(3)=\frac14.
\]

At \(d=40\),

\[
n\approx \frac{d^2}{4}=400.
\]

In [ ]:
KAPPA_GAUSSIAN = 3.0
ALPHA_STAR = 1.0 / 4.0

print({
    "kappa": KAPPA_GAUSSIAN,
    "alpha_star": ALPHA_STAR,
    "d40_predicted_n": int(ALPHA_STAR * 40**2),
})

## 5. Generate Gaussian instances

In [ ]:
def gaussian_instance(d, n, rng):
    return rng.normal(size=(n, d))

## 6. Solver policy

Each instance is attempted with available solvers in sequence.

Preferred order:

```text
CLARABEL
↓ retry if unresolved
SCS
```

A trial is classified as:

- `SAT` for a feasible solver status;
- `UNSAT` for an infeasible solver status;
- `UNRESOLVED` for solver exceptions or statuses that establish neither result.

The complete attempt record is retained.

In [ ]:
FEASIBLE_STATUSES = {
    cp.OPTIMAL,
    cp.OPTIMAL_INACCURATE,
}

INFEASIBLE_STATUSES = {
    cp.INFEASIBLE,
    cp.INFEASIBLE_INACCURATE,
}

def available_solver_order():
    installed = set(cp.installed_solvers())
    preferred = [s for s in ("CLARABEL", "SCS") if s in installed]
    if not preferred:
        raise RuntimeError("Neither CLARABEL nor SCS is available.")
    return preferred

SOLVER_ORDER = available_solver_order()
print("Solver order:", SOLVER_ORDER)

## 7. Ellipsoid-fitting SDP with retry

For each instance, solve for symmetric \(R\) satisfying

\[
x_i^\top R x_i=1
\]

for every sampled point together with the positive-semidefinite constraint.

An unresolved attempt is retried with the next solver.

In [ ]:
def solve_with_solver(problem, solver):
    if solver == "SCS":
        problem.solve(
            solver=solver,
            verbose=False,
            eps=1e-5,
            max_iters=50000,
        )
    else:
        problem.solve(
            solver=solver,
            verbose=False,
        )

def classify_status(status):
    if status in FEASIBLE_STATUSES:
        return "SAT"
    if status in INFEASIBLE_STATUSES:
        return "UNSAT"
    return "UNRESOLVED"

def ellipsoid_fit_reading(X, solver_order=SOLVER_ORDER):
    X = np.asarray(X, dtype=float)
    n, d = X.shape

    R = cp.Variable((d, d), symmetric=True)
    constraints = [R >> 0]
    constraints += [cp.quad_form(X[i], R) == 1 for i in range(n)]
    problem = cp.Problem(cp.Minimize(0), constraints)

    attempts = []

    for solver in solver_order:
        try:
            solve_with_solver(problem, solver)
            reading = classify_status(problem.status)

            attempts.append({
                "solver": solver,
                "status": str(problem.status),
                "reading": reading,
                "exception": None,
            })

            if reading in {"SAT", "UNSAT"}:
                return {
                    "reading": reading,
                    "final_solver": solver,
                    "final_status": str(problem.status),
                    "attempts": attempts,
                }

        except Exception as exc:
            attempts.append({
                "solver": solver,
                "status": None,
                "reading": "UNRESOLVED",
                "exception": f"{type(exc).__name__}: {exc}",
            })

    final = attempts[-1] if attempts else {}
    return {
        "reading": "UNRESOLVED",
        "final_solver": final.get("solver"),
        "final_status": final.get("status"),
        "attempts": attempts,
    }

## 8. Quick validation reading

In [ ]:
rng = np.random.default_rng(SEED)
X_check = gaussian_instance(d=6, n=6, rng=rng)
check = ellipsoid_fit_reading(X_check)

print("reading:", check["reading"])
print("final solver:", check["final_solver"])
print("final status:", check["final_status"])
print("attempts:")
for attempt in check["attempts"]:
    print(" ", attempt)

## 9. Sweep constraint density

For each

\[
\alpha=\frac{n}{d^2},
\]

the notebook records SAT, UNSAT, and UNRESOLVED separately.

Two derived quantities are computed:

\[
\text{fraction resolved}
=
\frac{\#\mathrm{SAT}+\#\mathrm{UNSAT}}{\#\mathrm{trials}},
\]

and, where at least one trial is resolved,

\[
\text{fraction SAT among resolved}
=
\frac{\#\mathrm{SAT}}
{\#\mathrm{SAT}+\#\mathrm{UNSAT}}.
\]

Thus solver failures never enter the denominator as UNSAT.

In [ ]:
def sweep_gaussian(d, alphas, trials, seed):
    rng = np.random.default_rng(seed)
    summary_rows = []
    trial_rows = []

    total = len(alphas) * trials
    completed = 0

    for alpha_target in alphas:
        n = max(1, int(round(alpha_target * d**2)))
        alpha = n / d**2

        counts = {
            "SAT": 0,
            "UNSAT": 0,
            "UNRESOLVED": 0,
        }

        for trial in range(1, trials + 1):
            X = gaussian_instance(d=d, n=n, rng=rng)
            result = ellipsoid_fit_reading(X)

            reading = result["reading"]
            counts[reading] += 1
            completed += 1

            retry_used = len(result["attempts"]) > 1

            print(
                f"{completed:>3}/{total}  "
                f"d={d:>2} n={n:>4} alpha={alpha:.4f}  "
                f"trial={trial}/{trials}  "
                f"{reading:<10}  "
                f"solver={result['final_solver']}  "
                f"status={result['final_status']}  "
                f"retry={retry_used}"
            )

            trial_rows.append({
                "d": d,
                "n": n,
                "alpha": alpha,
                "trial": trial,
                "reading": reading,
                "final_solver": result["final_solver"],
                "final_status": result["final_status"],
                "retry_used": retry_used,
                "attempts_json": json.dumps(result["attempts"]),
            })

        resolved = counts["SAT"] + counts["UNSAT"]
        fraction_resolved = resolved / trials
        fraction_unresolved = counts["UNRESOLVED"] / trials
        fraction_sat_resolved = (
            counts["SAT"] / resolved
            if resolved > 0
            else np.nan
        )

        summary_rows.append({
            "d": d,
            "n": n,
            "alpha": alpha,
            "trials": trials,
            "sat_count": counts["SAT"],
            "unsat_count": counts["UNSAT"],
            "unresolved_count": counts["UNRESOLVED"],
            "resolved_count": resolved,
            "fraction_resolved": fraction_resolved,
            "fraction_unresolved": fraction_unresolved,
            "fraction_sat_among_resolved": fraction_sat_resolved,
        })

    return pd.DataFrame(summary_rows), pd.DataFrame(trial_rows)

readings, trial_log = sweep_gaussian(
    d=D,
    alphas=ALPHAS,
    trials=TRIALS,
    seed=SEED,
)

## 10. Summary table

The key columns distinguish the three outcomes explicitly.

In [ ]:
display(
    readings[
        [
            "d",
            "n",
            "alpha",
            "trials",
            "sat_count",
            "unsat_count",
            "unresolved_count",
            "fraction_resolved",
            "fraction_sat_among_resolved",
        ]
    ]
)

## 11. Trial-level solver log

This table makes retries and solver statuses inspectable.

In [ ]:
display(
    trial_log[
        [
            "d",
            "n",
            "alpha",
            "trial",
            "reading",
            "final_solver",
            "final_status",
            "retry_used",
        ]
    ]
)

## 12. Plot resolved SAT readings

This plot answers:

> Among trials for which the solver established SAT or UNSAT, what fraction were SAT?

Unresolved trials are excluded from this fraction and shown separately in the next plot.

In [ ]:
fig_sat, ax = plt.subplots(figsize=(9, 5.5))

plot_data = readings.dropna(subset=["fraction_sat_among_resolved"])

ax.plot(
    plot_data["alpha"],
    plot_data["fraction_sat_among_resolved"],
    marker="o",
    linewidth=2,
    label="Fraction SAT among resolved trials",
)

ax.axvline(
    ALPHA_STAR,
    linestyle="--",
    linewidth=2,
    label=r"Theoretical boundary $\alpha_\star(3)=1/4$",
)

ax.set_xlabel(r"Constraint density $\alpha=n/d^2$")
ax.set_ylabel("Fraction SAT among resolved trials")
ax.set_ylim(-0.05, 1.05)
ax.set_title(
    f"Resolved ellipsoid-fitting readings — d={D}, "
    f"{TRIALS} trials per density"
)
ax.legend()
ax.grid(alpha=0.2)

plt.show()

## 13. Plot unresolved readings

This plot exposes solver difficulty rather than folding it into UNSAT.

In [ ]:
fig_unresolved, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    readings["alpha"],
    readings["fraction_unresolved"],
    marker="o",
    linewidth=2,
    label="Fraction unresolved",
)

ax.axvline(
    ALPHA_STAR,
    linestyle="--",
    linewidth=2,
    label=r"Theoretical boundary $\alpha_\star(3)=1/4$",
)

ax.set_xlabel(r"Constraint density $\alpha=n/d^2$")
ax.set_ylabel("Fraction unresolved")
ax.set_ylim(-0.05, 1.05)
ax.set_title(
    f"Solver resolution relative to the phase boundary — d={D}"
)
ax.legend()
ax.grid(alpha=0.2)

plt.show()

## 14. Compact computational reading

This cell reports the strongest claims directly supported by the run.

In [ ]:
below = readings[readings["alpha"] < ALPHA_STAR]
at_or_above = readings[readings["alpha"] >= ALPHA_STAR]

below_sat = int(below["sat_count"].sum())
below_trials = int(below["trials"].sum())
below_unresolved = int(below["unresolved_count"].sum())

print("Below theoretical Gaussian boundary:")
print(f"  SAT: {below_sat}/{below_trials}")
print(f"  unresolved: {below_unresolved}/{below_trials}")

print("\nAt or above theoretical Gaussian boundary:")
print(f"  SAT: {int(at_or_above['sat_count'].sum())}")
print(f"  UNSAT: {int(at_or_above['unsat_count'].sum())}")
print(f"  unresolved: {int(at_or_above['unresolved_count'].sum())}")

## 15. Save outputs

The summary, full solver log, and both plots are written as repository-ready artifacts.

In [ ]:
OUTPUT_DIR = Path("../outputs/2608-27372")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_csv = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_summary_v3.csv"
trial_csv = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_trial_log_v3.csv"
sat_png = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_resolved_sat_v3.png"
unresolved_png = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_unresolved_v3.png"

readings.to_csv(summary_csv, index=False)
trial_log.to_csv(trial_csv, index=False)

fig_sat.savefig(sat_png, dpi=180, bbox_inches="tight")
fig_unresolved.savefig(unresolved_png, dpi=180, bbox_inches="tight")

print("Saved:")
print(summary_csv)
print(trial_csv)
print(sat_png)
print(unresolved_png)

## 16. Engineering reading

The notebook now preserves the distinction

\[
\boxed{
\text{SAT}
\neq
\text{UNSAT}
\neq
\text{UNRESOLVED}
}
\]

through the complete workflow.

The theoretical object

\[
\alpha_\star(\kappa)
\]

specifies the asymptotic phase boundary.

The computation then returns a reading for each sampled instance. Solver difficulty is itself recorded as an unresolved computational state rather than silently converted into an UNSAT result.

For Gaussian coordinates,

\[
\boxed{
\kappa=3
\rightarrow
\alpha_\star(3)=\frac14
\rightarrow
\text{SDP computation}
\rightarrow
\text{SAT / UNSAT / unresolved reading}
}
\]